In [5]:
import os
import sys
from dotenv import load_dotenv
from utils.helper_functions import *
from utils.evaluate_rag import *

load_dotenv()

path="data/hyde_rag.pdf"

In [6]:
class HyDERetriever:
    def __init__(self,file_path,chunk_size=500,chunk_overlap=100):
        self.llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.6,max_completion_tokens=5000)
        self.embeddings=OpenAIEmbeddings(model='text-embedding-3-small')
        self.chunk_size=chunk_size
        self.chunk_overlap=chunk_overlap
        self.vectorstore=encode_pdf(file_path,self.chunk_size,self.chunk_overlap)

        self.HyDEPrompts=PromptTemplate(
            input_variables=["query","chunk_size"],
            template="""
                        You are an expert writer.
                        Given the following question, write a concise, factual document that would likely answer it.
                        The document should resemble a passage from a textbook, article, or knowledge base.
                        Do not mention that this is a hypothetical document.
                        Do not include phrases like "I think" or "As an AI".
                        the document size has be exactly {chunk_size} characters.

                    Question:
                    {query}
                    Hypothetical Document:
                    """
                    )
        self.HyDEChain=self.HyDEPrompts|self.llm

    def generate_hypothetical_documents(self,query:str):
        input_variable={"query":query,"chunk_size":self.chunk_size}
        return self.HyDEChain.invoke(input_variable).content

    def retrieve(self,query:str,k=3):
        hypothetical_docs=self.generate_hypothetical_documents(query)
        similar_docs=self.vectorstore.similarity_search(hypothetical_docs,k)
        return similar_docs,hypothetical_docs

        

In [8]:
retriever=HyDERetriever(path)
query="What happens when HyDE uses smaller instruction models like FLAN-T5 or Cohere?"
results,hypothetical_docs=retriever.retrieve(query)

print(f"{results}\n")
print(hypothetical_docs)

[Document(id='4b3d0a10-e337-4d1d-b6c2-e0f2d1463110', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-12-21T01:43:04+00:00', 'author': '', 'keywords': '', 'moddate': '2022-12-21T01:43:04+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/hyde_rag.pdf', 'total_pages': 11, 'page': 5, 'page_label': '6'}, page_content='on TREC DL19/20.\n5.1 Effect of Different Generative Models\nIn Table 4, we show HyDE using other\ninstruction-following language models. In\nparticular, we consider a 52-billion Cohere\nmodel ( command-xlarge-20221108) and a\n11-billion FLAN model ( FLAN-T5-xxl; Wei\net al. (2022)). 2 Generally, we observe that all\n2Model sizes are from https://crfm.stanford.edu/\nhelm/v1.0/?models.\nModel DL19 DL20\nContriever 44.5 42.1\nContrieverFT 62.1 63.2\nHyDE\nw/ Contriever\nw/ Flan-T5 (11b) 48.9 52.9'), Doc

In [9]:
docs_content=[doc.page_content for doc in results]

show_related_docs(docs_content)

Context:1
on TREC DL19/20.
5.1 Effect of Different Generative Models
In Table 4, we show HyDE using other
instruction-following language models. In
particular, we consider a 52-billion Cohere
model ( command-xlarge-20221108) and a
11-billion FLAN model ( FLAN-T5-xxl; Wei
et al. (2022)). 2 Generally, we observe that all
2Model sizes are from https://crfm.stanford.edu/
helm/v1.0/?models.
Model DL19 DL20
Contriever 44.5 42.1
ContrieverFT 62.1 63.2
HyDE
w/ Contriever
w/ Flan-T5 (11b) 48.9 52.9


Context:2
models bring improvement to the unsupervised
Contriever, with larger models bringing larger
improvements. At the time when this paper is
written, the Cohere model is still experimental
without much detail disclosed. We can only
tentatively hypothesize that training techniques
may have also played some role in the performance
difference.
5.2 HyDE with Fine-tuned Encoder
To begin with, HyDE with ﬁne-tuned encoder is
not the intended usage: HyDE is more powerful


Context:3
To begin with, Hy